# Recorrido por un EDA

Complemento de las páginas del curso *Fundamentos del EDA* y *Cómo: explorar un conjunto de datos*.

Por defecto se ejecuta sobre una **cohorte inventada de una clínica de lípidos** (el estudio de la variante LDLR de Mia) en la que se han plantado algunos problemas. Intenta encontrarlos antes de leer la última celda. Para usar tus propios datos, define `DATA_PATH` y `TARGET` más abajo.

El objetivo de explorar es entender los datos y generar preguntas, no confirmar nada. Necesita `numpy`, `pandas`, `matplotlib`, `seaborn`.

In [ ]:
# ---- Parameters: change these and re-run ----
DATA_PATH = None           # e.g. "my_data.csv"; the made-up demo dataset is used if None
TARGET = "ldl_mmol"       # the outcome column you care about (None to skip the outcome-focused plots)
SPLIT_BY = "statin"        # a categorical column to split subgroup plots by (None to skip)
DATE_COLUMNS = ["sample_date"]   # columns to parse as dates (empty list if none)
MAX_CATEGORIES = 12        # columns with at most this many distinct values are treated as categories
SEED = 0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def make_demo(n=300):
    """Made-up lipid-clinic cohort with some problems planted in it (see the last cell)."""
    rng = np.random.default_rng(SEED)
    clinic = rng.choice([f"C{i}" for i in range(1, 11)], n)
    statin = rng.choice(["no", "yes"], n, p=[0.6, 0.4])
    carrier_status = rng.choice(["non-carrier", "carrier"], n)
    age = rng.normal(50, 14, n).clip(18)
    days = rng.integers(0, 120, n)
    boost = np.where((carrier_status == "carrier") & (statin == "no"), 0.9, 0.0)
    y = 4.2 + boost - ((age - 55) / 25) ** 2 - 0.004 * days + rng.normal(0, 0.5, n)
    df = pd.DataFrame({
        "clinic": clinic, "statin": statin, "carrier_status": carrier_status, "age": age.round(0),
        "sample_date": pd.Timestamp("2024-06-01") + pd.to_timedelta(days, unit="D"),
        "ldl_mmol": y.round(1),
    })
    missing = (rng.random(n) < 0.04) | ((df["age"] < 32) & (rng.random(n) < 0.6)) | ((df["clinic"] == "C7") & (rng.random(n) < 0.5))
    df.loc[missing, "ldl_mmol"] = np.nan
    df.loc[17, "ldl_mmol"] = -3.2       # a typo
    df.loc[88, "ldl_mmol"] = 39.0       # a decimal slip
    return df


df = pd.read_csv(DATA_PATH, parse_dates=DATE_COLUMNS) if DATA_PATH else make_demo()

datetime_cols = df.select_dtypes("datetime").columns.tolist()
categorical = [c for c in df.columns if c not in datetime_cols
               and (not pd.api.types.is_numeric_dtype(df[c]) or df[c].nunique() <= MAX_CATEGORIES)]
numeric = [c for c in df.columns if c not in datetime_cols + categorical]
print(f"{df.shape[0]} rows x {df.shape[1]} columns")
print("numeric:", numeric, "| categorical:", categorical, "| dates:", datetime_cols)
df.head()

## 1. Calidad de los datos: rangos, duplicados, datos faltantes

Lee primero las columnas `min` y `max`: ¿hay algo imposible o sospechoso? Después pregúntate quién falta, porque los valores faltantes que dependen de otras variables no pueden descartarse sin pensarlo.

In [ ]:
print(df.describe(include="number").T[["count", "mean", "std", "min", "max"]].round(2))
print("\nduplicate rows:", df.duplicated().sum())

missing = df.isna().mean().sort_values(ascending=False)
print("\nshare missing:\n", missing[missing > 0].round(3))
fig, ax = plt.subplots(figsize=(6, 2 + 0.3 * len(missing)))
missing.plot.barh(ax=ax)
ax.set_xlabel("fraction missing")
plt.show()

if TARGET:
    print(f"\nAverage of the numeric columns, by whether {TARGET} is missing:")
    print(df.groupby(df[TARGET].isna())[numeric].mean().round(1))
    for cat in categorical:
        share = df.groupby(cat)[TARGET].apply(lambda s: s.isna().mean()).round(2)
        print(f"\nShare of {TARGET} missing, by {cat}:\n", share.to_string())

## 2. Una variable cada vez

Busca valores imposibles, puntos aislados muy alejados, acumulaciones y dos jorobas.

In [ ]:
df[numeric].hist(bins=30, figsize=(10, 3 * ((len(numeric) + 1) // 2)), layout=((len(numeric) + 1) // 2, 2))
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, max(len(categorical), 1), figsize=(4 * max(len(categorical), 1), 3))
for ax, cat in zip(np.atleast_1d(axes), categorical):
    df[cat].value_counts(dropna=False).plot.bar(ax=ax, title=cat)
plt.tight_layout()
plt.show()

## 3. Pares de variables

Los diagramas de dispersión revelan curvas, valores atípicos y grupos que un coeficiente de correlación oculta. Los valores extremos estiran los ejes, así que quizá quieras dejarlos aparte (solo para dibujar) una vez que los hayas detectado: pon `PLOT_RANGE` en el rango plausible de tu variable de respuesta.

In [ ]:
PLOT_RANGE = (0, 10)   # plausible range of the outcome, for plotting only; None to plot everything

plot_df = df
if TARGET and PLOT_RANGE:
    plot_df = df[df[TARGET].between(*PLOT_RANGE) | df[TARGET].isna()]
    print(f"Plotting {len(plot_df)} of {len(df)} rows (outcome outside {PLOT_RANGE} left out of the plots only).")

sns.pairplot(plot_df, vars=numeric[:6], hue=SPLIT_BY, corner=True)
plt.show()

## 4. La variable de respuesta por grupo y la estructura de subgrupos

Las cajas resumen; los puntos muestran cuántos datos hay detrás de cada caja. Colorear por una segunda categoría (`SPLIT_BY`) muestra si el patrón es el mismo en todos los subgrupos.

In [ ]:
if TARGET:
    for cat in categorical:
        if cat == TARGET:
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        hue = SPLIT_BY if SPLIT_BY and SPLIT_BY != cat else None
        sns.boxplot(data=plot_df, x=cat, y=TARGET, hue=hue, ax=ax)
        if hue is None:
            sns.stripplot(data=plot_df, x=cat, y=TARGET, color="black", alpha=0.3, ax=ax)
        plt.show()

    for col in numeric:
        if col == TARGET:
            continue
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.scatterplot(data=plot_df, x=col, y=TARGET, hue=SPLIT_BY, alpha=0.6, ax=ax)
        plt.show()

## 5. Tendencias a lo largo del tiempo

In [ ]:
if TARGET:
    for date_col in datetime_cols:
        ordered = plot_df.sort_values(date_col)
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(ordered[date_col], ordered[TARGET], ".", alpha=0.5)
        ax.plot(ordered[date_col], ordered[TARGET].rolling(30, min_periods=10).mean(), color="red", label="rolling mean")
        ax.set_title(f"{TARGET} over {date_col}")
        ax.legend()
        plt.show()

## Qué hacer con lo que has visto

Anota lo que has notado *como preguntas*. Todo lo que hayas encontrado mirando es una hipótesis que comprobar con datos nuevos, no un resultado. Corrige los errores claros en los datos en origen si puedes; si no, exclúyelos y dilo, indicando el número de filas afectadas.

---

**Spoiler: lo que se ha plantado en los datos de demostración**

- Un valor de LDL imposible (negativo) y un valor 10 veces mayor que parece un error de coma decimal: errores de datos, visibles en la tabla de mínimos y máximos y en el histograma.
- Valores de LDL faltantes que no son aleatorios: son más frecuentes en pacientes jóvenes y en la clínica C7. Descartar esas filas eliminaría discretamente a los pacientes jóvenes del análisis.
- El efecto de la variante que eleva el LDL solo existe en pacientes que no toman estatinas (las estatinas lo enmascaran): es invisible en la comparación global y evidente al separar por uso de estatinas. Lo encontraste mirando, así que es una pista que contrastar con datos nuevos, no un efecto confirmado.
- Una relación curva con la edad (el LDL alcanza su máximo en la mediana edad): un ajuste con una recta no la detectaría.
- Una deriva descendente con la fecha de la muestra, que podría deberse a un cambio en el ensayo o en la práctica clínica.

Los siguientes pasos dependen de tu objetivo: consulta las páginas del curso sobre datos faltantes y sobre predicción vs. causalidad.